# CS461 讽刺检测项目

这个笔记本用于所有实验：数据探索、模型训练、评估

**流程**：
1. 数据探索
2. 训练基线模型（逻辑回归 + TF-IDF）
3. 尝试进阶模型（SVM、LSTM等）
4. 保存最佳模型
5. 记录所有结果（用于报告）


## 第一步：加载库和数据


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# 加载数据
train_df = pd.read_csv('train.csv')
valid_df = pd.read_csv('valid.csv')
test_df = pd.read_csv('test.csv')

print(f"训练集: {len(train_df)} 条")
print(f"验证集: {len(valid_df)} 条")
print(f"测试集: {len(test_df)} 条")

# 查看前几行
train_df.head()


## 第二步：数据探索


In [ ]:
# 标签分布
print("标签分布:")
print(train_df['label'].value_counts())
print(f"\n讽刺比例: {train_df['label'].mean()*100:.2f}%")

# 可视化
train_df['label'].value_counts().plot(kind='bar')
plt.title('训练集标签分布')
plt.xlabel('标签 (0=非讽刺, 1=讽刺)')
plt.ylabel('数量')
plt.xticks(rotation=0)
plt.show()

# 文本长度
train_df['length'] = train_df['text'].str.len()
print(f"\n平均文本长度: {train_df['length'].mean():.1f} 字符")

# 查看样例
print("\n讽刺样例:")
for text in train_df[train_df['label']==1]['text'].head(3):
    print(f"  - {text}")
    
print("\n非讽刺样例:")
for text in train_df[train_df['label']==0]['text'].head(3):
    print(f"  - {text}")


## 第三步：训练基线模型 - 逻辑回归 + TF-IDF

目标：F1 > 0.70


In [ ]:
# 简单预处理：转小写
train_texts = train_df['text'].str.lower()
valid_texts = valid_df['text'].str.lower()
test_texts = test_df['text'].str.lower()

# TF-IDF特征提取
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(train_texts)
X_valid = vectorizer.transform(valid_texts)
X_test = vectorizer.transform(test_texts)

print(f"特征维度: {X_train.shape[1]}")

# 训练逻辑回归
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, train_df['label'])

# 验证集评估
y_pred_valid = model.predict(X_valid)
print("\n验证集性能:")
print(classification_report(valid_df['label'], y_pred_valid))
print(f"F1 Score: {f1_score(valid_df['label'], y_pred_valid):.4f}")

# 测试集评估
y_pred_test = model.predict(X_test)
print("\n测试集性能:")
print(classification_report(test_df['label'], y_pred_test))
print(f"Accuracy:  {accuracy_score(test_df['label'], y_pred_test):.4f}")
print(f"F1 Score:  {f1_score(test_df['label'], y_pred_test):.4f}")


## 第四步：保存模型

保存最佳模型用于最终提交


In [ ]:
import joblib

# 保存模型和向量化器
joblib.dump(model, 'models/model_weights.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')

print("✅ 模型已保存到 models/ 文件夹")


---

## 🔄 下一步改进方向

在下面添加新的代码单元格尝试：

1. **更多模型**：
   - SVM
   - Random Forest
   - Naive Bayes
   
2. **更好的特征**：
   - Word2Vec / GloVe
   - 不同的n-gram组合
   - 手工特征（标点、大写等）

3. **深度学习**：
   - LSTM
   - BiLSTM
   - CNN

4. **集成方法**：
   - 投票集成
   - Stacking

记得记录所有实验结果用于报告！
